# DataCoolie Fabric Spark Sample
Run file + delta stages from `fabric_use_cases.json` inside a Fabric notebook.

In [ ]:
%pip install "/lakehouse/default/Files/libraries/datacoolie-0.1.0-py3-none-any.whl" --force-reinstall
# %pip install datacoolie  # in the future

In [ ]:
notebookutils.session.restartPython()

In [ ]:
from datacoolie.engines.spark_engine import SparkEngine
from datacoolie.core.models import DataCoolieRunConfig
from datacoolie.metadata.file_provider import FileProvider
from datacoolie.orchestration.driver import DataCoolieDriver
from datacoolie.platforms.fabric_platform import FabricPlatform

# REPLACE <ROOT_PATH> with the path where you want to run this sample in your lakehouse.
# Make sure it has a Files subfolder for logs and metadata.
ROOT_PATH = "abfss://DataCoolie_DEV_WS@onelake.dfs.fabric.microsoft.com/lh_bronze.Lakehouse"

# Update this to where you stored fabric_use_cases.json in your lakehouse.
METADATA_PATH = f"{ROOT_PATH}/Files/metadata/fabric_use_cases.json"

# Keep initial execution narrow for first validation.
STAGE = "read_file,load_delta"

# Logs written through FabricPlatform under this Files folder.
BASE_LOG_PATH = f"{ROOT_PATH}/Files/logs/datacoolie"

try:
    spark_session = spark  # type: ignore[name-defined]
except NameError as exc:
    raise RuntimeError(
        "This sample must run inside a Fabric notebook where `spark` is preloaded."
    ) from exc

platform = FabricPlatform()
engine = SparkEngine(spark_session, platform=platform)
metadata = FileProvider(config_path=METADATA_PATH, platform=platform)
run_config = DataCoolieRunConfig(max_workers=8)

with DataCoolieDriver(
    engine=engine,
    metadata_provider=metadata,
    base_log_path=BASE_LOG_PATH,
    config=run_config,
) as driver:
    result = driver.run(stage=STAGE)

print(
    "Result:",
    {
        "total": result.total,
        "succeeded": result.succeeded,
        "failed": result.failed,
        "skipped": result.skipped,
    },
)